# 24장 트랜스포머 직접 만들어 보기

1.1.) 트랜스포머는 기존의 RNN, LSTM과 달리, **문장 내 단어들 간의 관계를 한꺼번에 처리한다** (계산 효율 향상 및 더 긴 문장도 효율적으로 처리)

1.2.) 트랜스포머의 주요 구성 요소 = **포지셔널 인코딩, 멀티헤드 어텐션, 정규화** + **가중치 행렬의 역할**
<br><br>


---

2.1.)기존 **RNN, LSTM**의 한계 = **문장을 앞에서부터 순서대로 따라가며** 단어 관계를 이해하는 구조 = 문장이 길어지면, 앞부분과의 연결을 파악하기 어려움..

2.2.) 이를 해결하기 위한 **Seq2Seq**의 인코더-디코더 구조 = 모든 정보를 하나의 문맥 벡터로 압축하면 정보가 소실되는 **병목 현상**

2.3.)이 Seq2Seq에 **어텐션을 추가**하면, 필요한 단어에 집중함으로써 정보를 더 효율적으로 처리 가능!!

2.4.)예를 들어, **Seq2Seq의 인코더-디코더 구조**를 남기되, 그 내부 처리과정에서 RNN, LSTM과 같은 순환신경망을 제거하고, **어텐션 매커니즘**만을 활용한 모델을 만든 것 = **트랜스포머**
<br><br>


---
3.1.) $멀티헤드 어텐션$의 원리

= **단어 임베딩** (단어를 컴퓨터가 인식할 수 있는 숫자로 표현) -> 이 행렬을 **쿼리-키-밸류로 나누고, 내적을 계산** (512차원 임베딩 벡터를 8개의 헤드로 쪼개서 64차원으로 나누고, 이들을 통해 내적 계산 및 원래 차원 복원) = **문맥 정보**를 반영할 수 있음.

3.2.)멀티헤드 어텐션의 장점 = **한 번의 여러 단어들 사이의 관계**를 파악 가능, 이렇게 얻은 다양한 저보를 하나로 합치면서 **더 풍부한 문맥**을 알아낼 수 있음, 또한 **처리속도**도 증가 (여러 헤드를 병렬 계산하기 때문)
<br><br>

---

4.1.)**$포지셔널 인코딩 벡터$** = 단어의 순서를 알려주는 것

-트랜스포머로 단어들을 순차적이 아니라 **한번에 처리하면, 효율은 좋아지지만, 단어의 위치 정보가 손실됨.**

-이를 해결하기 위해 포지셔널 인코딩이 사용됨 = **단어의 위치 정보를 추가해주는 과정**
<br><br>
4.2)맥락

-**어텐션** = 문장의 시작과 끝만 알려줄 뿐, 단어의 순서에 대한 정보는 주지 않음. 따라서, **단어의 위치 정보는 사라짐**

-이를 해결하기 위해, **임베딩 행렬과 똑같은 크기의 행렬을 만들고**, 어순을 정리함. 그리고 나서, **임베딩 행렬과 어순 정리 행렬을 더해줌**

-이렇게 하면, **위치 정보를 포함한 입력 행렬**을 만들 수 있고, 이것이 트랜스포머의 입력으로 사용됨.
<br><br>
4.3.)예시 = 어순을 정리한 새 행렬은 어떻게 만드는가? = **사인/코사인 함수의 주기**를 이용함

-기본 사인 그래프의 변화에 따라, [커피, 한잔, 어때]의 번호를 매겨보자. (S11, S21, S31) = 각 토큰의 첫번째 벡터(**홀수**)는 이렇게 **사인 그래프 값으로 채워짐**

-사인 그래프 (-1~+1)로만 긴 토큰을 채운다면, 언젠가 겹치는 값들이 나타나게 됨.

-따라서, **각 토큰의 짝수번째 벡터**는 **코사인 그래프 값으로 채움**

-512차원의 벡터를 채우기 위해서는, 각 삼각함수의 **주기**를 점진적으로 늘림 = 마치 삼각함수 그래프를 확대하듯이 **긴 주기의 삼각함수**가 만들어지고, 이를 바탕으로 **모든 위치의 단어에 대한 포지셔널 인코딩 벡터**를 만들 수 있음.
<br><br>

4.4.)포지셔널 인코딩 함수 공식

-**pos** = 단어의 위치 (e.g., 커피 (pos=0), 한잔 (pos=1), 어때 (pos=2))

-**i** =임베딩 벡터 안에서 해당 숫자가 몇 번째 차원에 있는가 (512차원에서 0~511번째 차원까지 총 512개의 숫자가 들어 있음)

-만약 i =1 (첫번째 숫자를 일컫음 = 차원 인덱스, **각 숫자의 위치**)

-**d^model** = **모델에서 사용하는 임베딩 벡터의 차원 수** (512라면, 각 단어가 512개의 숫자로 표현된다는 뜻)
<br><br>



---

5.1.)**$레이어 정규화$**

-활성화 함수에 전달되는 입력값이 한쪽으로 치우쳐 있다면, 가중치를 어떻게 조정해야 하는지 알려주는 **변화량**이 극적으로 크거나 작아지게됨

-이는 학습을 불안정하게 하여, 결국 모델이 제대로 수렴하지 못하게 함.

-따라서, **(평균 0, 분산1)의 표준 정규 분포로 바꿔서 정규화 해야함**

-이를 통해, **가중치 업데이트가 안정적으로 이루어지며, 모델 학습이 원활하게 됨**

-서로 다른 형태를 보이던 입력값들은 모두 표준 정규분포로 바꿀 수는 없음. 이 경우, 입력값의 특징들이 희석되어, 모델이 충분히 복잡한 관계를 학습하지 못함

-이를 해결하기 위해, **감마**(정규화된 값의 스케일 즉 높이를 조절)와 **베타**(그래프 좌우 이동 조절)를 도입함

-텐서플로에서는 **LayerNormalization 레이어**를 통해, 입력 벡터의 마지막 차원 (임베딩 벡터)의 평균/분산을 자동 계산하여, 감마/베타를 학습 가능한 형태로 유지해, 값을 적절히 조절해줌 (자동으로 해주기에, 이 레이어만 호출하면 됨)

-이를 통해, 모델이 학습을 진행하면서, 스스로 감마/베타 값을 업데이트하여, **가장 성능이 좋은 정규화 상태**를 자연스럽게 찾아냄. (정규화 과정 자동화 및 학습 안정성 향상
<br><br>


---

6.1.)**$잔차 연결$**

-잔차 연결이란, **멀티헤드 어텐션의 결과에 원래의 입력값을 더해주는 과정**이다.

-이를 통해, 딥러닝 모델이 깊어질 때 발생할 수 있는, 기울기 소실 문제를 완화해준다. (역전파 과정에서 기울기가 점점 작아서 학습이 어려워지는 문제를 방지하여, **학습을 안정화**)

-순서 = 레이어 정규화로 각 토큰의 임베딩을 **정규화** --> **멀티헤드 어텐션** 수행 --> 그 결과에 원래의 입력값을 더하는 **잔차 연결**

-잔차 연결을 수행하는 코드 = **Add()레이어**
<br><br>



---

7.1.)**$피드 포워드 신경망$**

-피드 포워드 신경망은 =두 개의 선형 변환 + 활성화 함수 (e.g. ReLU)로 구성됨.

-첫번째 선형 변환 = 입력 토큰의 **차원**을 증가시킴

-활성화 함수 적용 (e.g., ReLu) (**비선형**)

-두번째 선형 변환으로 = 원래 차원으로 되돌림


-이러한 피드 포워드 신경망 구조를 거치면, 모델은 **각 토큰에 대한 추가적인 정보를 학습**하는 거임

-**멀티헤드 어텐션** = 입력 토큰간 상호작용 모델링에는 좋으나, **각 토큰을 개별적으로 처리하는 능력은 부족**할 수 있음

-**피드 포워드 신경망을 사용하여, 각 토큰을 독립적으로 처리하여 변환**할 수 있음

-다시 말해, **단순한 선형 변환으로 얻기 어려운 복잡한 패턴을 학습시킬 수 있게 만들어, 표현력을 향상**시키는 것

<br><br>


---

8.1.)**$마스크드 어텐션$**

-아직 결정되지 않은 미래 단어를 볼 수 없게 하는 장치

-예측하려는 단어보다 오른쪽 (아직 생성되지 않은 미래 단어)에 대한 위치는 참고하지 않고, **이미 확정된 왼쪽 단어들로만 다음 단어를 예측하도록 하는 것**
<br><br>

-그림 24.5 예시

-디코더는 sos 토큰으로 문장 예측 시작 (**예측 결과 = sos**)

-**두 번째 단어 예측할 때는, 첫번째 토큰 sos를 참조하지만, 나머지 토큰은 못 봄**

-이 상태에서 디코더는 커피를 예측함 (**디코더 입력 = sos // 예측 결과 = 커피**)

-세 번째 단어 예측할 때는, sos/커피만 참고하여 예측하게 됨. (나머지는 가려져 있어야 함)

-이 상태에서 디코더는 한잔을 예측함 (**디코더 입력 = sos, 커피 // 예측 결과 = 한잔**)
<br><br>


8.2.)마스크드 어텐션을 적용하면, 디코더 내부에는 **마스크 행렬**이 생성됨.

-대각선 기준 왼쪽 아래 (이미 결정된 단어 위치)는 정상적으로 참조 가능하나

-오른쪽 위 부분 (미래 단어 위치)는 **-infinite**로 표시되어서, 소프트맥스 계산시 해당 위치에 어텐션 가중치를 0에 가깝게 만듦. = 즉 **참조 불가능한 상태를 의미**
<br><br>

8.3.)이를 통해, 디코더는 아직 등장하지 않은 단어를 엿보지 않고, **오직 이미 결정된 단어들과 입력 문장 정보를 바탕으로 다음 단어를 예측하는, 자연스러운 문장 생성 과정을 학습하게 됨.**

<br><br>



###피드 포워드 신경망 예시

In [ ]:
#피드 포워드 신경망 (FNN) 차원 설정
d_model = 128 #임베딩 벡터 차원 = 128
d_ff = 512 #피드 포워드 내부 차원 (임베딩 차원보다 큰 값)

#피드 포워드 신경망 구현
#1.첫번째 Dense 레이어: 차원을 d_ff로 확장하고, ReLU 활성화 함수 적용
ffn_intermediate =  Dense(d_ff, activation='relu')(attention_output_with_residual)

#2.두번째 Dense 레이어: 다시 원래의 차원 (d_model)로 축소
ffn_output = Dense(d_model)(ffn_intermediate)

#잔차 연결 적용
ffn_output_with_residual = Add()([attention_output_with_residual, ffn_output])

#레이어 정규화 적용
ffn_output_normalized = LayerNormalization()(ffn_output_with_residual)

###마스크드 행렬 예시

1.1)reshape와 tile을 이용해 이 행렬을 (batch_size, 1, seq_len, seq_len) 형태로 맞춰줌

1.2.)이를 통해 배치 내 모든 문장에 동일한 형태의 마스크를 적용

1.3.)이어서 -1e9로 치환하여, 소프트맥스 연산시 거의 0에 가까운 확률을 갖도록 만듦

1.4.)이를 통해, 해당 위치 (미래 단어)를 마치 볼 수 없는 영역인 것처럼 인식하게 됨

In [ ]:
import tensorflow as tf
import numpy as np

# seq_len 가정 = 여기서는 예시로 seq_len =4로 가정
seq_len = 4
batch_size = 2 #배치 크기도 예시로 2개 문장 처리로 가정

#마스크드 어텐션을 적용하는 경우라고 가정 (masked=True)
#1. 대각선 이하가 1, 그 위가 0인 행렬 생성
mask = tf.linalg.bang_part(tf.ones((seq_len, seq_len)), -1, 0)

#2. 현재 mask는 (4,4) 형태이므로, (1,1,4,4) 형태로 reshape
mask = tf.reshape(mask, (1, 1, seq_len, seq_len)

#3. 배치 크기만큼 tile을 이용해 확장 (batch_size, 1, seq_len, seq_len)
mask = tf.tile(mask [batch_size, 1, 1, 1])

#여기까지 만들어진 마스크로 numpy로 변환해 출력
mask_np = mask.numpy()
print("마스크 행렬 (1과 0으로 구성된 하삼각 형태):")
print(mask_np[0,0,:,:]) #첫번째 배치의 마스크 출력

#4. 0이었던 부분 (미래 단어 위치)를 -무한대로 설정
#실제 콛에서는 -1e9으로 대체해, 사실상 소프트맥스 계산시 0에 가깝게 만드는 효과
mask = mask*-1e9
mask_np = mask.numpy()

print("-무한대로 치환한 마스크 행렬:")
print(mask_np[0,0,:,:])

#출력 결과는 1이었던 위치는 그대로 (실제로 어텐션 계산시 이 부분의 값이 남아 있음)
#0이었던 위치는 -1e9로 치환되어 소프트맥스 시 거의 0에 가까운 확률을 줌

## 트랜스포머 인코더를 활용한 긍정 부정 예측


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D, LayerNormalization, Dropout, Add, Input
from tensorflow.keras.optimizers import Adam
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 깃허브에 준비된 데이터를 가져옵니다.
!git clone https://github.com/taehojo/data.git

# CSV 파일 로드
dataframe = pd.read_csv('./data/sentiment_data.csv')

# 데이터와 라벨 추출
sentences = dataframe['sentence'].tolist()
labels = dataframe['label'].tolist()

# 임베딩 벡터 크기와 최대 문장 길이 설정
embedding_dim = 128
max_len = 10

# 토크나이저 초기화 및 텍스트를 시퀀스로 변환
tokenizer = tf.keras.preprocessing.text.Tokenizer()
tokenizer.fit_on_texts(sentences)
sequences = tokenizer.texts_to_sequences(sentences)
word_index = tokenizer.word_index

# 패딩을 사용하여 시퀀스 길이를 동일하게 맞춤
data = tf.keras.preprocessing.sequence.pad_sequences(sequences, maxlen=max_len, padding='post')

# 데이터셋을 훈련 세트와 검증 세트로 분리
X_train, X_val, y_train, y_val = train_test_split(data, labels, test_size=0.2, random_state=42)

# 포지셔널 인코딩 함수
##max_len = 처리할 수 있는 문장의 최대 길이 (토큰 개수)
##pos_enc = max_len * d_model 형태의 2차원 배열을 만들어서, pos(위치)와 i(차원)에 대한 포지셔널 인코딩 값을 저장함
##짝수/홀수 차원에 따라 각각 사인과 코사인을 쓰도록 차원 인덱스 i를 2씩 증가시킴
def get_positional_encoding(max_len, d_model):
    pos_enc = np.zeros((max_len, d_model))
    for pos in range(max_len):
        for i in range(0, d_model, 2):
            pos_enc[pos, i] = np.sin(pos / (10000 ** (2 * i / d_model)))
            if i + 1 < d_model:
                pos_enc[pos, i + 1] = np.cos(pos / (10000 ** (2 * (i + 1) / d_model)))
    return pos_enc

# 포지셔널 인코딩 생성 (계산하는 코드)
##embedding_dim = 단어 임베딩의 차원 수 (우리가 설정한 임베딩 벡터 크기인 512차원을 말함)
positional_encoding = get_positional_encoding(max_len, embedding_dim)

# 멀티헤드 어텐션 레이어 (마스크 미사용)
class MultiHeadSelfAttentionLayer(tf.keras.layers.Layer):
    def __init__(self, num_heads, key_dim):
        super(MultiHeadSelfAttentionLayer, self).__init__()
        self.mha = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)
        self.norm = LayerNormalization()

    def call(self, x):
        attn_output = self.mha(query=x, value=x, key=x)
        attn_output = self.norm(attn_output + x)
        return attn_output

# 모델 설정
inputs = Input(shape=(max_len,))

# 1. 임베딩 레이어
embedding_layer = Embedding(input_dim=len(word_index) + 1, output_dim=embedding_dim)
embedded_sequences = embedding_layer(inputs)

# 2. 포지셔널 인코딩 추가 #앞서 생성한 포지셔널 인코딩을 임베딩 벡터에 더하여 **위치 정보를 포함하는 입력 행렬을 만듦**
embedded_sequences_with_positional_encoding = embedded_sequences + positional_encoding

# 3. 첫 번째 멀티헤드 어텐션 (마스크 없음)
attention_layer = MultiHeadSelfAttentionLayer(num_heads=8, key_dim=embedding_dim)
attention_output = attention_layer(embedded_sequences_with_positional_encoding)

# 4. 잔차 연결
attention_output_with_residual = Add()([embedded_sequences_with_positional_encoding, attention_output])

# 5. GlobalAveragePooling1D
##시퀀스 전체 단어 벡터를 평균내어, 하나의 문장 벡터로 만드는 역할
pooled_output = GlobalAveragePooling1D()(attention_output_with_residual)

# 6. 피드 포워드 네트워크
##Dense(d_ff, activation='relu')레이어를 통해, 차원을 확장하고 비선형 변환시킴
##이를 통해, 멀티헤드 어텐션을 거친 토큰 벡터를 더 풍부한 표현으로 만드는 것
##다시 말해, 단순한 선형 변환으로 얻기 어려운 복잡한 패턴을 학습시킬 수 있게 만들어, 표현력을 향상시키는 것

dense_layer = Dense(128, activation='relu')(pooled_output)
dropout_layer = Dropout(0.5)(dense_layer)
output_layer = Dense(1, activation='sigmoid')(dropout_layer)

#학습된 모델을 거쳐 -> 토큰화 -> 패딩과정 -> 확률값을 통해 긍정/부정을 판단.
model = Model(inputs=inputs, outputs=output_layer)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.fit(X_train, np.array(y_train), epochs=10, batch_size=16, validation_data=(X_val, np.array(y_val)))

# 새 문장 예측
sample_texts = ["I absolutely love this!", "I can't stand this product"]
sample_sequences = tokenizer.texts_to_sequences(sample_texts)
sample_data = tf.keras.preprocessing.sequence.pad_sequences(sample_sequences, maxlen=max_len, padding='post')
predictions = model.predict(sample_data)

for i, text in enumerate(sample_texts):
    print(f"Text: {text}")
    print(f"Prediction: {'Positive' if predictions[i] > 0.5 else 'Negative'}")


Epoch 1/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - accuracy: 0.5035 - loss: 0.8958 - val_accuracy: 0.9950 - val_loss: 0.2407
Epoch 2/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.9790 - loss: 0.1240 - val_accuracy: 1.0000 - val_loss: 0.0028
Epoch 3/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.9935 - loss: 0.0445 - val_accuracy: 0.9975 - val_loss: 0.0044
Epoch 4/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.9945 - loss: 0.0265 - val_accuracy: 0.9950 - val_loss: 0.0196
Epoch 5/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.9978 - loss: 0.0176 - val_accuracy: 1.0000 - val_loss: 0.0016
Epoch 6/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.9981 - loss: 0.0114 - val_accuracy: 1.0000 - val_loss: 9.9025e-04
Epoch 7/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.9998 - loss: 0.0013 - val_accuracy: 1.0000 - val_loss: 0.0015
Epoch 8/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.9954 - loss: 0.0102 - val_

## 전체 트랜스 포머 실행하기

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D, LayerNormalization, Dropout, Add, Input
from tensorflow.keras.optimizers import Adam
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 깃허브에 준비된 데이터를 가져옵니다.
!git clone https://github.com/taehojo/data.git

# CSV 파일 로드
dataframe = pd.read_csv('./data/sentiment_data.csv')

# 데이터와 라벨 추출
sentences = dataframe['sentence'].tolist()
labels = dataframe['label'].tolist()

# 임베딩 벡터 크기와 최대 문장 길이 설정
embedding_dim = 128
max_len = 10

# 토크나이저 초기화 및 텍스트를 시퀀스로 변환
tokenizer = tf.keras.preprocessing.text.Tokenizer()
tokenizer.fit_on_texts(sentences)
sequences = tokenizer.texts_to_sequences(sentences)
word_index = tokenizer.word_index

# 패딩을 사용하여 시퀀스 길이를 동일하게 맞춤
data = tf.keras.preprocessing.sequence.pad_sequences(sequences, maxlen=max_len, padding='post')

# 데이터셋을 훈련 세트와 검증 세트로 분리
X_train, X_val, y_train, y_val = train_test_split(data, labels, test_size=0.2, random_state=42)

# 포지셔널 인코딩 함수
def get_positional_encoding(max_len, d_model):
    pos_enc = np.zeros((max_len, d_model))
    for pos in range(max_len):
        for i in range(0, d_model, 2):
            pos_enc[pos, i] = np.sin(pos / (10000 ** (2 * i / d_model)))
            if i + 1 < d_model:
                pos_enc[pos, i + 1] = np.cos(pos / (10000 ** (2 * (i + 1) / d_model)))
    return pos_enc

# 포지셔널 인코딩 생성
positional_encoding = get_positional_encoding(max_len, embedding_dim)

# 멀티헤드 어텐션 레이어(마스크 지원)
class MultiHeadSelfAttentionLayer(tf.keras.layers.Layer):
    def __init__(self, num_heads, key_dim, masked=False):
        super(MultiHeadSelfAttentionLayer, self).__init__()
        self.mha = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)
        self.norm = LayerNormalization()
        self.masked = masked

    def call(self, x):
        if self.masked:
            batch_size = tf.shape(x)[0]
            seq_len = tf.shape(x)[1]
            mask = tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)
            mask = tf.reshape(mask, (1, 1, seq_len, seq_len))
            mask = tf.tile(mask, [batch_size, 1, 1, 1])
            mask = (1-mask) * -1e9
            attn_output = self.mha(query=x, value=x, key=x, attention_mask=mask)
        else:
            attn_output = self.mha(query=x, value=x, key=x)
        attn_output = self.norm(attn_output + x)
        return attn_output

# 모델 설정
inputs = Input(shape=(max_len,))

# 1. 임베딩 레이어
embedding_layer = Embedding(input_dim=len(word_index) + 1, output_dim=embedding_dim)
embedded_sequences = embedding_layer(inputs)

# 2. 포지셔널 인코딩 추가
embedded_sequences_with_positional_encoding = embedded_sequences + positional_encoding

# 3. 첫 번째 멀티헤드 어텐션 (마스크 없음)
attention_layer = MultiHeadSelfAttentionLayer(num_heads=8, key_dim=embedding_dim)
attention_output = attention_layer(embedded_sequences_with_positional_encoding)

# 4. 잔차 연결
attention_output_with_residual = Add()([embedded_sequences_with_positional_encoding, attention_output])

# 5. 마스크드 멀티헤드 어텐션 (마스크 있음)
masked_attention_layer = MultiHeadSelfAttentionLayer(num_heads=8, key_dim=embedding_dim, masked=True)
masked_attention_output = masked_attention_layer(attention_output_with_residual)

# 6. 잔차 연결
masked_attention_output_with_residual = Add()([attention_output_with_residual, masked_attention_output])

# 7. GlobalAveragePooling1D
pooled_output = GlobalAveragePooling1D()(masked_attention_output_with_residual)

# 8. 피드 포워드 네트워크
dense_layer = Dense(128, activation='relu')(pooled_output)
dropout_layer = Dropout(0.5)(dense_layer)
output_layer = Dense(1, activation='sigmoid')(dropout_layer)

model = Model(inputs=inputs, outputs=output_layer)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.fit(X_train, np.array(y_train), epochs=10, batch_size=16, validation_data=(X_val, np.array(y_val)))

# 새 문장 예측
sample_texts = ["I absolutely love this!", "I can't stand this product"]
sample_sequences = tokenizer.texts_to_sequences(sample_texts)
sample_data = tf.keras.preprocessing.sequence.pad_sequences(sample_sequences, maxlen=max_len, padding='post')
predictions = model.predict(sample_data)

for i, text in enumerate(sample_texts):
    print(f"Text: {text}")
    print(f"Prediction: {'Positive' if predictions[i] > 0.5 else 'Negative'}")

# 여기서 seq_len=4, batch_size=2로 예를 들어 마스크 행렬을 생성하고 출력해보는 예제
seq_len_example = 4
batch_size_example = 2
mask_example = tf.linalg.band_part(tf.ones((seq_len_example, seq_len_example)), -1, 0)
mask_example = tf.reshape(mask_example, (1, 1, seq_len_example, seq_len_example))
mask_example = tf.tile(mask_example, [batch_size_example, 1, 1, 1])

print("\n예시 마스크 행렬:")
print(mask_example.numpy()[0, 0, :, :])

mask_example = (1-mask_example) * -1e9
print("\n-∞로 치환한 마스크 행렬:")
print(mask_example.numpy()[0, 0, :, :])


Epoch 1/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 7s 41ms/step - accuracy: 0.5228 - loss: 0.9467 - val_accuracy: 0.9650 - val_loss: 0.3677
Epoch 2/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.9737 - loss: 0.1644 - val_accuracy: 1.0000 - val_loss: 0.0072
Epoch 3/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 35ms/step - accuracy: 0.9921 - loss: 0.0409 - val_accuracy: 1.0000 - val_loss: 0.0015
Epoch 4/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.9931 - loss: 0.0289 - val_accuracy: 0.9975 - val_loss: 0.0050
Epoch 5/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 33ms/step - accuracy: 0.9967 - loss: 0.0246 - val_accuracy: 1.0000 - val_loss: 0.0011
Epoch 6/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 4s 35ms/step - accuracy: 0.9950 - loss: 0.0194 - val_accuracy: 1.0000 - val_loss: 7.9798e-04
Epoch 7/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 4s 36ms/step - accuracy: 0.9957 - loss: 0.0211 - val_accuracy: 1.0000 - val_loss: 4.8506e-04
Epoch 8/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 4s 37ms/step - accuracy: 0.9989 - loss: 0.0094 - 